# Geolocation Exploratory Data Analysis (EDA)

## Objective
Analyze the Olist geolocation dataset to evaluate coordinate boundaries, identify geographic outliers outside Brazil, assess missing values and duplicates, and understand zip code prefix mappings.

## Dataset
`olist_geolocation_dataset.csv`

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

df_geo = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")
df_geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


## Data Structure & Integrity
Review total records, feature types, missing values, and duplicate rows.

In [4]:
print(f"Total Rows: {df_geo.shape[0]:,}")
print(f"Total Columns: {df_geo.shape[1]}")
print("\n--- Columns Info ---")
df_geo.info()

print("\n--- Missing Values ---")
print(df_geo.isnull().sum())

print(f"\nDuplicate Rows: {df_geo.duplicated().sum():,}")

Total Rows: 1,000,163
Total Columns: 5

--- Columns Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB

--- Missing Values ---
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicate Rows: 261,831


## Coordinate Boundaries & Outlier Detection
Evaluate latitude and longitude ranges to check for geographic anomalies outside Brazil's bounding box.

In [5]:
print("Latitude Summary:")
print(df_geo['geolocation_lat'].describe())

print("\nLongitude Summary:")
print(df_geo['geolocation_lng'].describe())

# Brazil geographic bounding box boundaries
# Latitude: 5.27 to -33.75 | Longitude: -34.79 to -73.98
lat_outliers = df_geo[(df_geo['geolocation_lat'] > 5.27) | (df_geo['geolocation_lat'] < -33.75)]
lng_outliers = df_geo[(df_geo['geolocation_lng'] > -34.79) | (df_geo['geolocation_lng'] < -73.98)]

print(f"\nLatitude Outliers Count: {len(lat_outliers)}")
print(f"Longitude Outliers Count: {len(lng_outliers)}")

Latitude Summary:
count    1.000163e+06
mean    -2.117615e+01
std      5.715866e+00
min     -3.660537e+01
25%     -2.360355e+01
50%     -2.291938e+01
75%     -1.997962e+01
max      4.506593e+01
Name: geolocation_lat, dtype: float64

Longitude Summary:
count    1.000163e+06
mean    -4.639054e+01
std      4.269748e+00
min     -1.014668e+02
25%     -4.857317e+01
50%     -4.663788e+01
75%     -4.376771e+01
max      1.211054e+02
Name: geolocation_lng, dtype: float64

Latitude Outliers Count: 31
Longitude Outliers Count: 37


## Zip Code Aggregation & Customer Relationship
Analyze unique zip code prefixes and multi-coordinate mapping behavior per prefix.

In [7]:
print("Unique Geolocation Zip Code Prefixes:", df_geo['geolocation_zip_code_prefix'].nunique())


zip_coords_count = df_geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].nunique()
print("\nCoordinates per Zip Code Summary:")
print(zip_coords_count.describe())

Unique Geolocation Zip Code Prefixes: 19015

Coordinates per Zip Code Summary:
       geolocation_lat  geolocation_lng
count     19015.000000     19015.000000
mean         37.828031        37.835656
std          49.929342        49.944594
min           1.000000         1.000000
25%           8.000000         8.000000
50%          22.000000        22.000000
75%          48.000000        48.000000
max         746.000000       745.000000


## Key Findings Summary
- Total Records & Integrity: 1,000,163 rows with 0 missing values across all columns.
- Duplicate Rows: 261,831 fully duplicate rows identified (requires deduplication during cleaning).
- Geographic Boundaries: Coordinate analysis detects minor outliers (31 lat, 37 lng) outside Brazil's geographic bounds that must be filtered out.
- Zip Code Mapping: 19,015 unique zip code prefixes exist with multi-coordinate records per prefix (averaging ~38 points per zip code), requiring spatial aggregation (`groupby.mean()`) during cleaning for accurate 1-to-1 customer joining.